# 🎙️ NeuTTS Air — Airi Wake Word Dataset Generator
### Voice Cloning TTS on Google Colab

This notebook generates synthetic `.wav` samples of **"Hello Airi"** using NeuTTS Air.
It clones voices from short reference audio clips to create highly realistic, diverse samples.

---

## 📋 How to use this notebook (read first!)

Follow the cells **in order, top to bottom**. Each cell has a clear heading telling you what it does.

**One-time setup (do this once per Colab session):**

1. ▶️ Run **Cell 1** — installs all software
2. 🔄 **Restart the session** (`Runtime → Restart session`) — required after installs
3. ▶️ Run **Cell 2** — sets your configuration
4. ▶️ Run **Cell 3** — downloads sample voices (or upload your own)
5. ▶️ Run **Cell 4** — loads the AI model
6. ▶️ Run **Cell 5** — generates your dataset
7. ▶️ Run **Cell 6** — downloads your files

> 💡 **GPU tip:** Go to `Runtime → Change runtime type → T4 GPU` before starting. It makes generation 10–50× faster and it is free!

---

## ⚙️ Cell 1 — Install Everything

**What this does:** Installs espeak-ng (a speech tool), neutts (the AI model package), and llama-cpp-python (needed internally by neutts).

**What you do after:** When it finishes and you see the `ACTION REQUIRED` box, go to `Runtime → Restart session`. That's it — you don't need to re-run this cell again.

> ⚠️ This cell takes **3–5 minutes** the first time. The large text output is normal — just let it run.

In [ ]:
import sys, subprocess
print(f'Python version: {sys.version}')
print()

# ── Step 1: espeak-ng (speech phonemizer — system tool) ──────────────────
print('Step 1/3: Installing espeak-ng (speech tool)...')
!apt-get update -qq
!apt-get install -y -qq espeak-ng
print('  espeak-ng done.')
print()

# ── Step 2: llama-cpp-python (required by neutts internally) ─────────────
# This is the biggest install — it compiles from source, takes 2-3 min.
print('Step 2/3: Installing llama-cpp-python (this takes 2-3 min, please wait)...')
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('  llama-cpp-python done.')
else:
    print('  llama-cpp-python FAILED. Error:')
    print(r.stderr[-1500:])
print()

# ── Step 3: neutts + audio libs ──────────────────────────────────────────
print('Step 3/3: Installing neutts and audio libraries...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'neutts', 'soundfile', 'scipy'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('  neutts + audio libs done.')
else:
    print('  neutts FAILED. Error:')
    print(r.stderr[-1500:])
print()

# ── Final check ───────────────────────────────────────────────────────────
import shutil
espeak_ok = shutil.which('espeak-ng') is not None
print('=' * 55)
print(f'  espeak-ng : {"OK" if espeak_ok else "MISSING — re-run this cell"}')
print()
if espeak_ok:
    print('  ALL INSTALLS COMPLETE!')
    print()
    print('  >>> ACTION REQUIRED <<<')
    print('  Go to: Runtime -> Restart session')
    print('  Then run Cell 2 and onwards (skip Cell 1).')
else:
    print('  Something went wrong. Please re-run this cell.')
print('=' * 55)


## 🔧 Cell 2 — Configuration

**What this does:** Sets up all the settings for your dataset.

**What you can change:**
- `WAKE_WORD` — the phrase you want to synthesize
- `TARGET_SAMPLES` — how many audio files you want (100 is a good start, 600 for full training)
- `MIN_DURATION` / `MAX_DURATION` — accepted length of each clip in seconds

**Everything else** — leave as-is unless you know what you're doing.

In [ ]:
import os, torch
from pathlib import Path

# ============================================================
#  YOU CAN CHANGE THESE VALUES
# ============================================================
WAKE_WORD      = 'Hello Airi'   # The phrase to synthesize
TARGET_SAMPLES = 100            # Start with 100; increase to 600 when working
MIN_DURATION   = 1.0            # Minimum clip length in seconds
MAX_DURATION   = 1.5            # Maximum clip length in seconds
BATCH_SIZE     = 10             # How many clips to generate per progress update
RANDOM_SEED    = 42             # Change this number each run for variety
# ============================================================

# These are set automatically — do not change
OUTPUT_DIR   = '/content/neutts_output'   # Where .wav files are saved
SAMPLES_DIR  = '/content/ref_voices'      # Where reference voice clips go
SAMPLE_RATE  = 24000                       # NeuTTS Air native output rate
BACKBONE_REPO = 'neuphonic/neutts-air-q4-gguf'
CODEC_REPO    = 'neuphonic/neucodec'

# Auto-detect GPU
if torch.cuda.is_available():
    BACKBONE_DEVICE = 'cuda'
    CODEC_DEVICE    = 'cuda'
    print(f'GPU detected: {torch.cuda.get_device_name(0)}')
    print('Great! Generation will be fast.')
else:
    BACKBONE_DEVICE = 'cpu'
    CODEC_DEVICE    = 'cpu'
    print('No GPU found — running on CPU (slower).')
    print('Tip: Runtime -> Change runtime type -> T4 GPU')

# Create output folders
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(SAMPLES_DIR).mkdir(parents=True, exist_ok=True)

print()
print('Configuration set!')
print(f'  Wake word      : {WAKE_WORD!r}')
print(f'  Target samples : {TARGET_SAMPLES}')
print(f'  Duration range : {MIN_DURATION}s to {MAX_DURATION}s')
print(f'  Output folder  : {OUTPUT_DIR}')


## 🎤 Cell 3 — Get Reference Voices

**What this does:** Downloads a set of sample voice clips from the NeuTTS repository. These voices are what the AI will clone when generating your wake word samples.

**You have two options:**

- **Option A (recommended for beginners):** Run the cell below — it auto-downloads sample voices
- **Option B (advanced):** Upload your own `.wav` files (3–15 seconds of clean speech each) by running the second code block instead

> The more different voices you have, the more diverse your dataset will be.

In [ ]:
# ── OPTION A: Download built-in sample voices (recommended) ──────────────
import shutil, subprocess
from pathlib import Path

REPO_DIR = '/tmp/neutts_repo'

# Remove any previous failed clone attempt
shutil.rmtree(REPO_DIR, ignore_errors=True)

print('Downloading sample voices from NeuTTS repository...')
print('(This downloads only the voice files, not the whole repo)')

result = subprocess.run(
    ['git', 'clone', '--depth=1', '--filter=blob:none', '--sparse',
     'https://github.com/neuphonic/neutts-air.git', REPO_DIR],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('Clone failed. Error:')
    print(result.stderr)
else:
    subprocess.run(
        ['git', '-C', REPO_DIR, 'sparse-checkout', 'set', 'samples'],
        capture_output=True
    )
    repo_samples = Path(REPO_DIR) / 'samples'
    copied = 0
    for f in repo_samples.glob('*'):
        shutil.copy(f, Path(SAMPLES_DIR) / f.name)
        if f.suffix == '.wav':
            print(f'  Copied voice: {f.name}')
            copied += 1
    print()
    print(f'Done! {copied} voice file(s) ready in {SAMPLES_DIR}')

# Show what we have
ref_wavs = list(Path(SAMPLES_DIR).glob('*.wav'))
print(f'Total reference voices available: {len(ref_wavs)}')
for w in ref_wavs:
    print(f'  - {w.name}')


In [ ]:
# ── OPTION B: Upload your own voice files (skip if you ran Option A) ─────
# Run this block INSTEAD of Option A if you want to use your own voice clips.
# Each file should be a .wav, 3-15 seconds, clear speech, minimal background noise.

from google.colab import files
from pathlib import Path

print('Please upload one or more .wav files (3-15 seconds of clear speech each).')
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = Path(SAMPLES_DIR) / fname
    dest.write_bytes(data)
    print(f'  Saved: {fname}')

ref_wavs = list(Path(SAMPLES_DIR).glob('*.wav'))
print(f'\nTotal reference voices: {len(ref_wavs)}')


## 📝 Cell 4 — Prepare Voice Registry

**What this does:** Each voice clip needs a text transcript (what the person is saying in it). This cell checks for transcript files. If any are missing, it uses Whisper AI to auto-transcribe them — so you don't have to type anything manually.

**What you do:** Just run it.

In [ ]:
from pathlib import Path

SAMPLES_PATH = Path(SAMPLES_DIR)
ref_wavs = sorted(SAMPLES_PATH.glob('*.wav'))

if not ref_wavs:
    raise FileNotFoundError(
        f'No .wav files found in {SAMPLES_DIR}!\n'
        'Please go back and run Cell 3 first.'
    )

print(f'Found {len(ref_wavs)} voice file(s). Checking for transcripts...')

# Check which ones are missing a .txt transcript
missing = [w for w in ref_wavs if not w.with_suffix('.txt').exists()]

if missing:
    print(f'{len(missing)} file(s) have no transcript — auto-transcribing with Whisper...')
    print('(Whisper is a free speech-to-text tool — this may take a moment)')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai-whisper'],
                   capture_output=True)
    import whisper
    whisper_model = whisper.load_model('base')
    for wav in missing:
        result = whisper_model.transcribe(str(wav))
        transcript = result['text'].strip()
        wav.with_suffix('.txt').write_text(transcript)
        print(f'  Transcribed {wav.name!r}: {transcript!r}')
else:
    print('All voices already have transcripts.')

# Build the registry list
REF_REGISTRY = []
for wav in ref_wavs:
    txt = wav.with_suffix('.txt')
    transcript = txt.read_text().strip() if txt.exists() else ''
    REF_REGISTRY.append({
        'name': wav.stem,
        'wav' : str(wav),
        'text': transcript,
    })

print()
print('Voice registry ready:')
for r in REF_REGISTRY:
    short_text = r['text'][:50] + '...' if len(r['text']) > 50 else r['text']
    print(f"  [{r['name']}] says: {short_text!r}")


## 📦 Cell 5 — Load the NeuTTS Air Model

**What this does:** Downloads and loads the NeuTTS Air AI model.

- First run: downloads ~400 MB (Q4 model) — takes a few minutes
- Later runs: loads from cache instantly

**What you do:** Just run it and wait.

In [ ]:
import shutil

# Safety check — make sure espeak-ng is still present after restart
if shutil.which('espeak-ng') is None and shutil.which('espeak') is None:
    raise EnvironmentError(
        'espeak-ng is not installed!\n'
        'Please go back to Cell 1 and run it again, then restart the session.'
    )
print(f'espeak-ng found at: {shutil.which("espeak-ng")}')

# Import neutts
try:
    from neutts import NeuTTS
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        'neutts is not installed!\n'
        'Please go back to Cell 1 and run it again, then restart the session.'
    )

print(f'Loading NeuTTS Air model...')
print(f'  Model  : {BACKBONE_REPO}')
print(f'  Device : {BACKBONE_DEVICE.upper()}')
print('(First run downloads ~400 MB — please wait)')
print()

tts = NeuTTS(
    backbone_repo=BACKBONE_REPO,
    backbone_device=BACKBONE_DEVICE,
    codec_repo=CODEC_REPO,
    codec_device=CODEC_DEVICE,
)

print('Model loaded successfully!')
print()

# Pre-encode all reference voices (done once, reused for all samples)
print('Pre-encoding reference voices (done once)...')
ENCODED_REFS = []
for ref in REF_REGISTRY:
    ref_codes = tts.encode_reference(ref['wav'])
    ENCODED_REFS.append({
        'name'      : ref['name'],
        'ref_codes' : ref_codes,
        'ref_text'  : ref['text'],
    })
    print(f"  Encoded: {ref['name']}")

print()
print(f'All {len(ENCODED_REFS)} voice(s) encoded and ready.')
print('You can now run Cell 5 to generate your dataset!')


## 🎙️ Cell 6 — Generate Your Dataset

**What this does:** Runs the generation loop. For each sample it:
1. Picks a random reference voice
2. Synthesizes the wake word
3. Applies random pitch shift, speed change, and emotion tone
4. Checks the clip length — rejects clips that are too short or too long
5. Saves accepted clips as `.wav` files

**What you do:** Run it and watch the progress. Each `OK` line is a saved sample. Each `SKIP` means the clip was the wrong length and was discarded — this is normal.

> ⏱️ **Time estimate:** GPU ~15–30 min for 600 samples. CPU ~2–5 hours.

In [ ]:
import numpy as np, random, time, soundfile as sf
from scipy import signal
from pathlib import Path

# Variation pools — the source of diversity in your dataset
EMOTION_STYLES = ['neutral', 'happy', 'sad', 'angry', 'surprised',
                  'fearful', 'disgusted', 'calm', 'excited']
SPEED_RANGE    = [0.85, 0.9, 1.0, 1.05, 1.1]
PITCH_RANGE    = [-3, -2, -1, 0, 1, 2, 3]  # semitones

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ── Audio helpers ────────────────────────────────────────────────────────
def change_pitch(audio, n_steps):
    if n_steps == 0: return audio
    ratio = 2 ** (n_steps / 12.0)
    return signal.resample(signal.resample(audio, int(len(audio) / ratio)), len(audio))

def change_speed(audio, speed):
    if speed == 1.0: return audio
    idx = np.round(np.arange(0, len(audio), speed)).astype(int)
    return audio[idx[idx < len(audio)]]

def add_emotion(audio, emotion):
    if emotion in ('happy', 'excited'):   audio = audio * 1.1
    elif emotion in ('sad', 'calm'):       audio = audio * 0.9
    elif emotion == 'angry':               audio = audio * 1.2
    elif emotion == 'fearful':
        tr = 1 + 0.1 * np.sin(2 * np.pi * 5 * np.arange(len(audio)) / SAMPLE_RATE)
        audio = audio * tr
    mx = np.abs(audio).max()
    return audio / mx * 0.95 if mx > 0 else audio


# ── Generation loop ──────────────────────────────────────────────────────
output_path      = Path(OUTPUT_DIR)
valid_samples    = 0
rejected_samples = 0
total_generated  = 0
epoch            = 1
start_time       = time.time()

print('=' * 60)
print('  AIRI WAKE WORD GENERATION — NeuTTS Air')
print('=' * 60)
print(f'  Wake word      : {WAKE_WORD!r}')
print(f'  Target samples : {TARGET_SAMPLES}')
print(f'  Duration range : {MIN_DURATION}s to {MAX_DURATION}s')
print(f'  Reference voices: {len(ENCODED_REFS)}')
print(f'  Device         : {BACKBONE_DEVICE.upper()}')
print('=' * 60)

while valid_samples < TARGET_SAMPLES:
    remaining  = TARGET_SAMPLES - valid_samples
    batch_size = min(BATCH_SIZE, remaining)
    print(f'\n--- Epoch {epoch} | {valid_samples}/{TARGET_SAMPLES} done | {remaining} to go ---')

    bv = br = 0
    for i in range(batch_size):
        ref     = random.choice(ENCODED_REFS)
        emotion = random.choice(EMOTION_STYLES)
        speed   = random.choice(SPEED_RANGE)
        pitch   = random.choice(PITCH_RANGE)

        try:
            wav   = tts.infer(WAKE_WORD, ref['ref_codes'], ref['ref_text'])
            audio = np.array(wav, dtype=np.float32)
            audio = change_speed(audio, speed)
            audio = change_pitch(audio, pitch)
            audio = add_emotion(audio, emotion)

            dur = len(audio) / SAMPLE_RATE
            if not (MIN_DURATION <= dur <= MAX_DURATION):
                br += 1
                print(f'  SKIP [{i+1:>2}/{batch_size}] {dur:.2f}s — outside {MIN_DURATION}–{MAX_DURATION}s window')
            else:
                ts    = int(time.time() * 1000)
                sign  = 'p' if pitch >= 0 else 'n'
                fname = (f"airi_{ts}_{i}_{ref['name'][:8]}_"
                         f"{emotion[:4]}_spd{speed:.2f}_pit{sign}{abs(pitch)}.wav")
                sf.write(str(output_path / fname), audio, SAMPLE_RATE)
                bv += 1
                valid_samples += 1
                print(f"  OK  [{i+1:>2}/{batch_size}] voice={ref['name']:<10} "
                      f"emo={emotion:<9} spd={speed:.2f} dur={dur:.2f}s")
        except Exception as e:
            br += 1
            print(f'  ERR [{i+1:>2}/{batch_size}] {e}')

        total_generated += 1

    rejected_samples += br
    epoch            += 1
    elapsed = time.time() - start_time
    rate    = valid_samples / elapsed if elapsed > 0 else 0
    eta     = (TARGET_SAMPLES - valid_samples) / rate if rate > 0 else float('inf')
    print(f'  Batch: {bv} saved, {br} skipped | '
          f'Total: {valid_samples}/{TARGET_SAMPLES} | '
          f'Elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min')


elapsed = time.time() - start_time
print('\n' + '=' * 60)
print('  GENERATION COMPLETE!')
print('=' * 60)
print(f'  Valid samples  : {valid_samples}')
print(f'  Rejected/skipped: {rejected_samples}')
print(f'  Success rate   : {valid_samples/total_generated*100:.1f}%')
print(f'  Total time     : {elapsed/60:.1f} minutes')
print(f'  Avg per sample : {elapsed/valid_samples:.2f}s')
print(f'  Files saved to : {OUTPUT_DIR}')
print('=' * 60)
print()
print('Next step: run Cell 7 to preview a sample, or Cell 8 to download everything.')


## 🔍 Cell 7 — Preview a Sample (Optional)

**What this does:** Plays a randomly chosen clip from your generated dataset right here in the notebook.

Run this to do a quick sanity check before downloading.

In [ ]:
from IPython.display import Audio, display
import random as _rnd
from pathlib import Path

wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))
if not wav_files:
    print('No .wav files found yet — run Cell 6 first to generate samples.')
else:
    sample = _rnd.choice(wav_files)
    print(f'Playing a random sample: {sample.name}')
    print(f'(Total files generated: {len(wav_files)})')
    display(Audio(str(sample)))


## 📥 Cell 8 — Download Your Dataset

**Important:** Colab sessions are temporary. Everything in `/content/` is deleted when the session ends. Download your files before closing the tab!

**Option A** — Downloads a `.zip` file directly to your computer.

**Option B** — Saves files to your Google Drive (safer, no risk of browser download timeout).

> 💡 For large datasets (600 files), Option B (Google Drive) is more reliable.

In [ ]:
# ── OPTION A: Download as zip directly to your computer ──────────────────
import zipfile
from google.colab import files
from pathlib import Path

zip_path  = '/content/airi_neutts_dataset.zip'
wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))

if not wav_files:
    print('No .wav files found — run Cell 6 first.')
else:
    print(f'Zipping {len(wav_files)} files...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in wav_files:
            zf.write(f, f.name)
    size_mb = Path(zip_path).stat().st_size / (1024 ** 2)
    print(f'Zip ready: {size_mb:.1f} MB')
    print('Starting download...')
    files.download(zip_path)


In [ ]:
# ── OPTION B: Save to Google Drive (recommended for large datasets) ───────
import shutil, os
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DEST = '/content/drive/MyDrive/airi_neutts_dataset'
os.makedirs(DRIVE_DEST, exist_ok=True)

wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))
if not wav_files:
    print('No .wav files found — run Cell 6 first.')
else:
    print(f'Copying {len(wav_files)} files to Google Drive...')
    for f in wav_files:
        shutil.copy(f, DRIVE_DEST)
    print(f'Done! All files saved to: {DRIVE_DEST}')
    print('You can find them in your Google Drive under: My Drive/airi_neutts_dataset/')
